# CNNs: Multiple Input and Output Channels

## 1. From Grayscale to Multi-Channel Inputs (RGB)
Previously, a grayscale image was represented as a 2D matrix:
$$I \in \mathbb{R}^{H \times W}$$

An RGB color image has **3 input channels** (Red, Green, and Blue), making it a 3D tensor:
$$I \in \mathbb{R}^{H \times W \times 3}$$

## 2. How a Single Filter Processes Multiple Channels
When processing a multi-channel input (like an RGB image), a single convolutional filter is no longer a 2D matrix—it becomes a **3D volume** whose depth matches the number of input channels ($C_{in}$).

* For a $3 \times 3$ spatial kernel on an RGB image ($C_{in} = 3$), one filter consists of **three separate $3 \times 3$ weight matrices** (one for Red, one for Green, and one for Blue).
* **Weights per filter:** $3 \times 3 \times 3 = 27\text{ weights}$ (plus 1 scalar bias $b$ for the entire filter).

### The Computation
At any spatial position $(r, c)$, the filter extracts a $3 \times 3 \times 3$ window from the input, computes a weighted sum for each channel independently, and adds them all together with a single bias:

$$Y[r, c] = \sum_{d=0}^{2} \sum_{i=0}^{2} \sum_{j=0}^{2} I[r+i, c+j, d] \cdot K[i, j, d] + b$$

Or more intuitively:
$$Y = S_R + S_G + S_B + b$$

* **Key Takeaway:** Even though the input has 3 channels, **one 3D filter produces only ONE 2D output feature map**. All input channels are collapsed/summed into a single output channel.

## 3. Creating Multiple Output Feature Maps
To detect multiple distinct features (e.g., horizontal edges, vertical edges, color blobs), a convolutional layer uses **multiple independent filters** ($C_{out}$).

* Each filter has its own set of 3D weights and its own bias.
* Each filter scans the input image independently to produce its own 2D feature map.
* Stacking these $C_{out}$ feature maps together forms the final 3D output tensor.

### Parameter Count Formula
For a convolutional layer with kernel size $K_h \times K_w$, input channels $C_{in}$, and output filters $C_{out}$:
* **Total Weights:** $(K_h \times K_w \times C_{in}) \times C_{out}$
* **Total Biases:** $C_{out}$
* **Total Trainable Parameters:** $(K_h \times K_w \times C_{in} + 1) \times C_{out}$

## 4. Worked Examples

### Example 1: Parameter Counting
**Setup:**
* Input image shape: $(100, 100, 3)$ (RGB)
* Kernel spatial size: $3 \times 3$
* Number of filters: $5$

**Calculations:**
1. **Weights in 1 filter:** $3 \times 3 \times 3 = 27\text{ weights}$
2. **Weights across all 5 filters:** $27 \times 5 = 135\text{ weights}$
3. **Total trainable parameters (including 1 bias per filter):** $135 + 5 = 140\text{ parameters}$

### Example 2: Single Window Calculation
**Setup:**
At a specific spatial location, a $3 \times 3 \times 3$ filter computes the channel-wise weighted sums:
* $S_R = 20$
* $S_G = -10$
* $S_B = 35$
* Filter bias $b = 5$

**Calculation:**
$$Y = 20 + (-10) + 35 + 5 = 50$$
The single output pixel value at this location in the feature map is **50**.

### Example 3: Output Tensor Shape
**Setup:**
* Input shape: $(100, 100, 3)$
* Kernel size: $3 \times 3$, Stride: $1$, Padding: $0$
* Number of filters: $5$

**Calculation:**
* **Spatial Height ($H_{out}$):** $100 - 3 + 1 = 98$
* **Spatial Width ($W_{out}$):** $100 - 3 + 1 = 98$
* **Output Depth ($C_{out}$):** $5$ (one 2D map per filter)

**Final Output Tensor Shape:** `(98, 98, 5)`


In [4]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [1]:
def correlate3d(image, kernel, padding=0, stride=1):
    # image: (H, W, C)
    # kernel: (Kh, Kw, C)
    # output: (Oh, Ow)
    pass

In [2]:
import numpy as np


def correlate2d_valid(
    image,
    kernel,
    padding: int = 0,
    stride: int = 1,
    pad_type: str = "constant"
):
    # 1. Validate parameters
    if not isinstance(stride, (int, np.integer)) or stride <= 0:
        raise ValueError("Stride must be a positive integer.")

    if not isinstance(padding, (int, np.integer)) or padding < 0:
        raise ValueError("Padding must be a non-negative integer.")

    if pad_type not in ("constant", "reflect", "edge"):
        raise ValueError(
            "Padding type must be 'constant', 'reflect', or 'edge'."
        )

    # 2. Convert inputs to float64
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)

    if image.ndim != 2 or kernel.ndim != 2:
        raise ValueError("Image and kernel must be 2D arrays.")

    kh, kw = kernel.shape

    if kh == 0 or kw == 0:
        raise ValueError("Kernel cannot be empty.")

    # 3. Apply padding
    if pad_type == "constant":
        padded = np.pad(
            image,
            pad_width=padding,
            mode="constant",
            constant_values=0.0
        )
    else:
        padded = np.pad(
            image,
            pad_width=padding,
            mode=pad_type
        )

    pih, piw = padded.shape

    if pih < kh or piw < kw:
        raise ValueError(
            "Kernel dimensions exceed padded image dimensions."
        )

    # 4. Calculate output dimensions
    oh = (pih - kh) // stride + 1
    ow = (piw - kw) // stride + 1

    # 5. Allocate output
    output = np.zeros((oh, ow), dtype=np.float64)

    # 6. Perform cross-correlation
    for out_r, r in enumerate(range(0, pih - kh + 1, stride)):
        for out_c, c in enumerate(range(0, piw - kw + 1, stride)):

            window = padded[r:r + kh, c:c + kw]

            output[out_r, out_c] = np.sum(window * kernel)

    return output

In [3]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def correlate2d_valid_fast(
    image,
    kernel,
    padding: int = 0,
    stride: int = 1,
    pad_type: str = "constant",
):
  if stride <= 0:
    raise ValueError("stride must be a positive integer.")
  if padding < 0:
    raise ValueError("padding must be non-negative.")

  img = np.asarray(image, dtype=np.float64)
  ker = np.asarray(kernel, dtype=np.float64)

  kh, kw = ker.shape
  if kh == 0 or kw == 0:
    raise ValueError("Kernel must be non-empty.")

  if padding > 0:
    if pad_type == "constant":
      img = np.pad(img, pad_width=padding, mode=pad_type, constant_values=0.0)
    else:
      img = np.pad(img, pad_width=padding, mode=pad_type)

  ih, iw = img.shape
  if kh > ih or kw > iw:
    raise ValueError("Kernel dimensions cannot exceed padded image dimensions.")

  windows = sliding_window_view(img, (kh, kw))[::stride, ::stride]
  return np.einsum("ijkl,kl->ij", windows, ker, optimize=True)